# RAG (Retrieval-Augmented Generation)
Basic pipeline: load documents → chunk → embed → store → retrieve → generate

In [ ]:
# Install dependencies
%pip install openai faiss-cpu tiktoken pypdf sentence-transformers

In [ ]:
import os
import faiss
import numpy as np
from pathlib import Path
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

## 1. Load Documents

In [ ]:
def load_text(file_path: str) -> str:
    """Load a plain text or PDF file."""
    path = Path(file_path)
    if path.suffix == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(file_path)
        return "\n".join(page.extract_text() for page in reader.pages)
    return path.read_text(encoding="utf-8")

# Example: replace with your own file
# raw_text = load_text("your_document.pdf")
raw_text = "Artificial intelligence (AI) is intelligence demonstrated by machines. " \
           "Machine learning is a subset of AI. Deep learning is a subset of machine learning. " \
           "RAG stands for Retrieval-Augmented Generation. It combines retrieval with generation."
print(f"Loaded {len(raw_text)} characters")

## 2. Chunk Text

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_text)
print(f"{len(chunks)} chunks created")
for i, c in enumerate(chunks):
    print(f"[{i}] {c[:80]}...")

## 3. Embed Chunks

In [ ]:
def get_embeddings(texts: list[str], model: str = "text-embedding-3-small") -> np.ndarray:
    response = client.embeddings.create(input=texts, model=model)
    return np.array([e.embedding for e in response.data], dtype=np.float32)

embeddings = get_embeddings(chunks)
print(f"Embedding shape: {embeddings.shape}")

## 4. Build Vector Store (FAISS)

In [ ]:
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f"Index contains {index.ntotal} vectors")

## 5. Retrieve Relevant Chunks

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[str]:
    query_embedding = get_embeddings([query])
    _, indices = index.search(query_embedding, top_k)
    return [chunks[i] for i in indices[0]]

query = "What is RAG?"
retrieved = retrieve(query)
print(f"Top {len(retrieved)} chunks for query: '{query}'")
for i, chunk in enumerate(retrieved):
    print(f"\n[{i+1}] {chunk}")

## 6. Generate Answer

In [ ]:
def generate(query: str, context_chunks: list[str]) -> str:
    context = "\n\n".join(context_chunks)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer using only the provided context. If the answer is not in the context, say so."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ]
    )
    return response.choices[0].message.content

answer = generate(query, retrieved)
print(f"Q: {query}")
print(f"A: {answer}")

## 7. Full Pipeline

In [ ]:
def rag(query: str) -> str:
    context = retrieve(query)
    return generate(query, context)

# Try your own questions here
print(rag("What is the relationship between AI and deep learning?"))